# 01 — Acquire and Understand the MyLA311 Dataset

**Question this notebook answers:** What is the MyLA311 dataset, and what does it actually contain?

This notebook does **not** clean, filter, map, or analyze. It only builds familiarity with the raw data.

## Where this data comes from

- **Source:** [MyLA311 Cases 2026](https://data.lacity.org/City-Infrastructure-Service-Requests/MyLA311-Cases-2026/2cy6-i7zn) on the LA Open Data Portal (`data.lacity.org`), maintained by the City of LA Information Technology Agency. Updated daily. License: CC0 (public domain).
- **How it got here:** `python shared/scripts/download_myla311.py 2026` downloads the full CSV export (~384 MB) into `shared/datasets/raw/`. Re-run it anytime to refresh; see the script for other years.
- **Important context:** LA migrated its 311 system to a new Salesforce-based platform in late 2025. Datasets from 2015–2025 use the old *"Service Request Data"* schema; 2026 onward uses the new *"Cases"* schema. Column names and category definitions changed, so **cross-era comparisons will require mapping work**. Full details in [`shared/references/myla311.md`](../references/myla311.md).

## Loading the data — why pandas?

**pandas** is the standard Python library for working with tabular data. Its core object, the `DataFrame`, is an in-memory table — think "SQL table you can manipulate programmatically," where every operation (filter, group, join) is a method call instead of a query.

We load the whole CSV into memory here because 1.2M rows is small by pandas standards (a few GB of RAM at worst) and it makes exploration instant. Later, when we work across multiple years, DuckDB will let us query files *without* loading them fully — but for schema familiarization, pandas is the simplest tool.

One flag worth explaining: `low_memory=False`. By default pandas reads big CSVs in chunks and guesses each column's type per-chunk, which can produce inconsistent types (and a warning). This flag makes it consider whole columns at once — slightly more RAM, but consistent types.

In [1]:
from pathlib import Path

import pandas as pd

RAW = Path("../datasets/raw/myla311_cases_2026.csv")

df = pd.read_csv(RAW, low_memory=False)

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")

1,209,648 rows x 34 columns


## Columns and data types

`df.dtypes` shows what type pandas inferred for each column. A quick decoder:

- `str` — text (pandas 3 shows this as `str`; older tutorials call it `object`)
- `int64` / `float64` — integers / floating-point numbers. **Watch out:** a numeric column containing *any* missing values gets promoted to `float64`, because the classic integer type can't represent "missing." That's why `CD` (council district) shows up as a float — it isn't really fractional.
- `bool` — True/False

Everything in a CSV is text on disk; these types are pandas' *guesses* from the values it saw.

In [2]:
df.dtypes

CaseNumber                     int64
CreatedDate                      str
UpdatedDate                      str
ActionTaken                      str
Owner                            str
RequestType                      str
Status                           str
RequestSource                    str
CreatedByUserOrganization        str
Anonymous                       bool
AssignTo                         str
ServiceDate                      str
ClosedDate                       str
DateServiceRendered              str
ReasonCode                       str
ResolutionCode                   str
Address                          str
HouseNumber                      str
Direction                        str
StreetName                       str
Suffix                           str
ZipCode                      float64
Latitude                     float64
Longitude                    float64
Location                     float64
TBMPage                      float64
TBMColumn                        str
T

## Dates

CSVs carry no type information, so dates arrive as strings like `01/01/2026 12:00:31 AM`. `pd.to_datetime` parses them into proper timestamp objects so we can compute ranges (and later, durations). We parse copies here rather than modifying `df` — this notebook leaves the raw data untouched.

In [3]:
created = pd.to_datetime(df["CreatedDate"], format="mixed")
closed = pd.to_datetime(df["ClosedDate"], format="mixed")
updated = pd.to_datetime(df["UpdatedDate"], format="mixed")

print(f"CreatedDate: {created.min()}  ->  {created.max()}")
print(f"ClosedDate:  {closed.min()}  ->  {closed.max()}")
print(f"UpdatedDate: {updated.min()}  ->  {updated.max()}")

CreatedDate: 2026-01-01 00:00:31  ->  2026-07-06 15:59:59
ClosedDate:  2026-01-01 00:08:57  ->  2026-07-06 15:59:59
UpdatedDate: 2026-01-01 00:09:01  ->  2026-07-06 15:59:59


The data runs from January 1, 2026 up to (nearly) the moment it was downloaded — this dataset is refreshed continuously, not in monthly batches.

## Request categories

`value_counts()` is the workhorse for categorical columns: it returns each distinct value with its frequency, sorted descending. (`nunique()` just counts distinct values.)

In [4]:
print(f"{df['RequestType'].nunique()} unique request types\n")
df["RequestType"].value_counts()

62 unique request types



RequestType
Information-Only                   280335
Item Pickups                       245814
Illegal Dumping Item Pickup        242533
Graffiti Removal                   120991
Service Not Complete                66182
                                    ...  
Trees or Pests in the Park             68
Bicycle Facilities Issues              58
Illegal Dumping Food Waste              6
Searchlight & Generator Permits         5
Public Right-of-Way Cleanup             1
Name: count, Length: 62, dtype: int64

Worth pausing on. The old system had ~12 request types; the new one has **62**, much more granular (e.g. `Illegal Dumping Item Pickup` vs `Illegal Dumping Food Waste`, four separate animal categories). Also notice:

- **`Information-Only` is the single largest category (~280K, 23%)** — people calling 311 to ask questions, not report problems. For maintenance analysis these are noise, but they say a lot about what 311 actually *is*: as much a call center as a work-order system.
- **`Service Not Complete` (~66K)** is intriguing — it appears to be follow-ups on requests that didn't get resolved. Potentially a direct signal for our recurrence questions.

## Status, source, and owning department

In [5]:
print("--- Status ---")
print(df["Status"].value_counts().to_string())
print(f"\n--- RequestSource (top 10 of {df['RequestSource'].nunique()}) ---")
print(df["RequestSource"].value_counts().head(10).to_string())
print(f"\n--- Owner: {df['Owner'].nunique()} distinct values, top 15 ---")
print(df["Owner"].value_counts().head(15).to_string())

--- Status ---
Status
Closed                 1019428
Reported                 65068
Workorder Created        51665
Cancelled                42755
New                      14729
In Progress               7100
Closed Ext-Referred       5914
Potential Duplicate       2576
Duplicate Confirm          413

--- RequestSource (top 10 of 18) ---
RequestSource
Call                        599584
Self Service                579384
recycLA Service Provider     12478
Driver Self Reported          6200
Mobile App                    3673
Council's Office              3491
Email                         3100
Voicemail                     1023
Chat                           351
Social                         122



--- Owner: 99 distinct values, top 15 ---


Owner
LASAN                                    617700
ITA                                      164613
OCB - Normal Cases                        72780
LSD                                       49645
LASAN - LSD                               49381
RAP - Construction                        46172
BSS - SMD                                 44016
BSL                                       25216
recycLA                                   25181
BSS - IED                                 24019
LADOT - District Operations               18984
BSS - UFD                                 15324
LADOT - Dockless Mobility Enforcement     11386
LASAN - CCD Staff                          9366
DPW                                        6742


## Geographic fields

In [6]:
print(f"Council districts (CD): {df['CD'].nunique()} -> {sorted(df['CD'].dropna().unique())}")
print(f"\nNeighborhood Councils (NCName): {df['NCName'].nunique()}")
print(f"\nArea Planning Commissions (APC): {df['APC'].nunique()}")
print(df["APC"].value_counts().to_string())
print(f"\nZIP codes: {df['ZipCode'].nunique()}")
print(f"Police precincts: {df['PolicePrecinct'].nunique()}")
print(f"\nLatitude range:  {df['Latitude'].min():.4f} to {df['Latitude'].max():.4f}")
print(f"Longitude range: {df['Longitude'].min():.4f} to {df['Longitude'].max():.4f}")

Council districts (CD): 16 -> [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0), np.float64(12.0), np.float64(13.0), np.float64(14.0), np.float64(15.0)]

Neighborhood Councils (NCName): 99

Area Planning Commissions (APC): 7


APC
Central APC              273587
South Los Angeles APC    213349
South Valley APC         205772
North Valley APC         162794
East Los Angeles APC     106419
West Los Angeles APC     105877
Harbor APC                44232

ZIP codes: 593
Police precincts: 22

Latitude range:  -38.0022 to 56.1304
Longitude range: -158.0085 to 174.7597


Three things to note:

1. **CD has 16 values, but LA has 15 council districts.** The extra is `0`, presumably "outside city limits / couldn't assign." Something to handle when we eventually clean.
2. **99 Neighborhood Councils** matches LA's certified NC count — this field looks trustworthy.
3. **The lat/lon ranges are garbage at the extremes.** LA sits around latitude 33.7–34.3, longitude -118.7 to -118.1. Values like latitude -38 or longitude +174 are geocoding failures. Most rows are fine, but *we cannot trust coordinates blindly* — a lesson to bank for when we start mapping.

## Missing data

`df.isna()` marks every missing cell; taking the column-wise mean turns that into a fraction missing. Knowing *where the holes are* tells us which columns we can actually rely on.

In [7]:
missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct

Location                     100.0
DateServiceRendered           99.5
ServiceDate                   98.1
ReasonCode                    91.2
HouseNumber                   81.7
Direction                     67.0
ActionTaken                   49.7
AssignTo                      34.6
Suffix                        22.4
StreetName                    19.0
NCName                         9.7
APC                            8.1
CDMember                       8.1
TBMPage                        8.0
TBMRow                         8.0
NC                             8.0
TBMColumn                      8.0
PolicePrecinct                 8.0
Address                        8.0
CD                             6.7
ResolutionCode                 6.4
ClosedDate                     6.3
Latitude                       5.4
Longitude                      5.4
ZipCode                        5.1
CreatedByUserOrganization      1.3
CreatedDate                    0.0
Anonymous                      0.0
RequestSource       

## A few raw rows

Transposing (`.T`) a small sample makes wide rows readable — 34 columns as rows instead of a horizontal scroll. `random_state` pins the random sample so the notebook shows the same rows on every run (reproducibility).

In [8]:
df.sample(3, random_state=42).T

,599474,1016605,830786
CaseNumber,3164606,3720935,3472346
CreatedDate,04/02/2026 12:24:43 PM,06/05/2026 02:14:15 PM,05/07/2026 04:57:16 PM
UpdatedDate,04/09/2026 06:47:43 AM,06/05/2026 02:14:15 PM,05/07/2026 04:57:17 PM
ActionTaken,SR Created,Information Provided,Information Provided
Owner,LASAN,LASAN,LASAN
RequestType,Item Pickups,Information-Only,Information-Only
Status,Closed,Closed,Closed
RequestSource,Call,Call,Call
CreatedByUserOrganization,Self Service,Self Service,Self Service
Anonymous,True,True,True


## Column-by-column guide

What each column represents, whether it looks useful for our research questions, and its caveats.

### Identity & lifecycle

| Column | What it is | Useful? | Caveats |
|---|---|---|---|
| `CaseNumber` | Unique case ID | ✅ join key, dedup | — |
| `CreatedDate` | When the case was opened | ✅ essential — time series, seasonality | none seen; 0% missing |
| `UpdatedDate` | Last system modification | maybe | reflects system activity, not service delivery |
| `ClosedDate` | When the case was closed | ✅ essential — resolution time = Closed − Created | 6% missing (open cases, mostly); "closed" ≠ "problem fixed" |
| `ServiceDate` / `DateServiceRendered` | When service was scheduled/performed | ❌ mostly not | **98–99% empty** — the new system barely populates these |
| `Status` | Case state (`Closed`, `Reported`, `Workorder Created`…) | ✅ | 84% are `Closed`; ~43K `Cancelled` and ~3K duplicate-flagged rows will need filtering decisions |
| `ActionTaken` | What the system/staff did | maybe | 50% missing |
| `ReasonCode` | Why (e.g. for cancellation) | ❌ mostly not | 91% missing |
| `ResolutionCode` | Coded outcome | ✅ potentially | 6% missing; need to learn the code vocabulary |

### What & who

| Column | What it is | Useful? | Caveats |
|---|---|---|---|
| `RequestType` | Category of request | ✅ the core categorical field | 62 granular types; not comparable 1:1 with pre-2026 categories |
| `RequestSource` | Channel (`Call`, `Self Service`, app…) | ✅ for reporting-bias questions | channel mix differs by neighborhood — a *bias lens*, not just metadata |
| `CreatedByUserOrganization` | Who entered it | maybe | |
| `Anonymous` | Reporter anonymity flag | marginal | |
| `Owner` | Responsible department/bureau | ✅ but messy | 96 distinct values incl. duplicates (`LASAN LSD` vs `LASAN - LSD`), person names, and a literal `TEST XX` — free-text-ish, needs normalization before use |
| `AssignTo` | Assigned crew/queue | maybe | 35% missing |

### Where

| Column | What it is | Useful? | Caveats |
|---|---|---|---|
| `Address`, `HouseNumber`, `Direction`, `StreetName`, `Suffix` | Address components | ✅ for corridor analysis | 8–82% missing depending on component; `Address` (8% missing) is the usable one |
| `ZipCode` | ZIP | ✅ coarse grouping | 5% missing |
| `Latitude` / `Longitude` | Geocoded point | ✅ essential for mapping | 5% missing **plus** out-of-bounds junk values |
| `Location` | Combined point field | ❌ | 100% empty in CSV export — ignore |
| `TBMPage` / `TBMColumn` / `TBMRow` | Thomas Guide map grid refs | ❌ | legacy paper-map indexing |
| `APC` | Area Planning Commission (7 regions) | ✅ coarse regions | |
| `CD` / `CDMember` | Council district number / member name | ✅ for CD comparisons | includes `CD=0`; member names go stale as politicians change |
| `NC` / `NCName` | Neighborhood Council id/name | ✅ our best neighborhood proxy | NCs are political boundaries, not "neighborhoods" exactly |
| `PolicePrecinct` | LAPD area | maybe | |

## What we learned, and questions this raises

**Learned:**
- 1.21M cases in just over six months of 2026 — LA is on pace for ~2.4M/year through this system.
- The dataset is genuinely rich: precise timestamps, 62 categories, coordinates, and four different geographic groupings (ZIP, NC, CD, APC).
- It is refreshed daily and licensed CC0 — we can do anything with it.
- The 311 system was replaced in late 2025; historical continuity is broken at the migration boundary.

**Surprises:**
- ~23% of all cases are `Information-Only` — 311 is as much a Q&A line as a maintenance intake.
- Illegal dumping (~243K) + item pickups (~246K) + graffiti (~121K) dwarf everything else operational. Our research focus areas are the system's biggest workload.
- Data quality is uneven in specific, documentable ways: junk coordinates, a `TEST XX` owner, near-empty service-date fields.

**Questions raised (added to `docs/research_questions.md` candidates):**
1. What distinguishes `Item Pickups` from `Illegal Dumping Item Pickup` operationally?
2. What does `Service Not Complete` mean exactly — and is it a ready-made recurrence signal?
3. Does `Closed` mean the problem was fixed, or just that the case was closed? (`ResolutionCode` vocabulary may answer this.)
4. Are `Information-Only` calls geographically skewed? Even "noise" may reveal who relies on 311.
5. How do the 62 new categories map onto the ~12 old ones, if we ever want pre-2026 trends?

**Next checkpoint candidates:** learn the `ResolutionCode` vocabulary; decide filtering rules (cancelled/duplicate/info-only); or start on a first research question with this data as-is.